# 4.2 指令微调 / Instruction tuning

预训练模型只会续写。看对话模板和损失掩码怎样把它变成会回答问题的助手，并亲手微调一次。

A pretrained model only continues text. See how a chat template and a loss mask turn it into an assistant that answers, and run a fine-tuning yourself.

讲解与推导见网站对应模块。每个练习先写代码，再运行紧随其后的检查单元。

The explanation and derivations are on the website. For each exercise, write the code, then run the check cell right after it.

## 1. 手写对话模板与损失掩码 / A chat template and a loss mask by hand

实现 ChatML 模板，并与一个真实的开放聊天模型（SmolLM2）自带的模板逐字符对比。再实现一个函数，标出哪些 token 属于回答、需要计算损失。测试会把标记为需要训练的 token 解码出来，确认正好是各条回答的内容加上结束标记。这个练习只下载分词器，不下载模型权重。

Implement the ChatML template and compare it character by character with the template that ships with a real open chat model (SmolLM2). Then implement a function that marks which tokens belong to a response and need a loss. The tests decode the tokens marked for training and confirm that they are exactly the content of each response plus the end marker. This exercise downloads only the tokenizer, not the model weights.

需要联网下载模型。/ Downloads a model.

In [ ]:
from transformers import AutoTokenizer


def chatml(messages, add_generation_prompt=False):
    """Format a conversation in ChatML, the template used by many open chat models.

    messages: list of {"role": "system" | "user" | "assistant", "content": str}
    Every message becomes   <|im_start|>ROLE\\nCONTENT<|im_end|>\\n
    With add_generation_prompt the string ends with   <|im_start|>assistant\\n
    so that the model continues by writing the assistant's reply.
    """
    # TODO 1
    raise NotImplementedError


def assistant_mask(tok, messages):
    """Which tokens should be trained on?

    Tokenize chatml(messages) with tok(text, add_special_tokens=False). Return (input_ids, mask), two
    lists of equal length. mask[i] is True for the tokens of every assistant message's CONTENT and
    its closing <|im_end|>, and False everywhere else (system and user turns, role headers, newlines).

    Hint: build the text piece by piece and tokenize each piece separately, so you know which
    pieces belong to the assistant.
    """
    # TODO 2
    raise NotImplementedError

In [ ]:
# 检查 / checks
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-135M-Instruct")   # a small open chat model that uses ChatML
msgs = [{"role": "system", "content": "You are concise."},
        {"role": "user", "content": "Hi"},
        {"role": "assistant", "content": "Hello!"},
        {"role": "user", "content": "What is 2+2?"}]

assert chatml(msgs[:2]) == "<|im_start|>system\nYou are concise.<|im_end|>\n<|im_start|>user\nHi<|im_end|>\n"
for n in (2, 3, 4):
    for gen in (False, True):
        ref = tok.apply_chat_template(msgs[:n], tokenize=False, add_generation_prompt=gen)
        assert chatml(msgs[:n], gen) == ref, f"differs from the model's own template ({n} messages, generation prompt {gen})"

# the control markers are single tokens, not ordinary text
assert len(tok("<|im_start|>", add_special_tokens=False).input_ids) == 1
assert len(tok("<|im_end|>", add_special_tokens=False).input_ids) == 1

conv = msgs + [{"role": "assistant", "content": "4"}]
ids, mask = assistant_mask(tok, conv)
assert len(ids) == len(mask) and ids == tok(chatml(conv), add_special_tokens=False).input_ids, "pieces must add up to the full text"
trained = tok.decode([i for i, m in zip(ids, mask) if m])
assert trained == "Hello!<|im_end|>4<|im_end|>", f"the trained tokens decode to {trained!r}"
assert sum(mask) < len(mask) / 2, "most of this conversation is context, not training target"
print("m42_template: all tests passed")
print(" ", repr(chatml(msgs, True)))

<details><summary><b>提示 1 / Hint 1</b></summary>

每条消息是 `f"<|im_start|>{role}\n{content}<|im_end|>\n"`，全部拼接起来。需要时在最后再加 `"<|im_start|>assistant\n"`。

Each message is `f"<|im_start|>{role}\n{content}<|im_end|>\n"`, all concatenated. When asked, append `"<|im_start|>assistant\n"` at the end.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

掩码的做法是把每条消息拆成三段分别分词，即角色头、内容加结束标记、最后的换行。只有角色为 assistant 的第二段标记为 True。

For the mask, split each message into three pieces and tokenize them separately, namely the role header, the content plus end marker, and the final newline. Only the second piece of an assistant message is marked True.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

```python
from transformers import AutoTokenizer


def chatml(messages, add_generation_prompt=False):
    text = "".join(f"<|im_start|>{m['role']}\n{m['content']}<|im_end|>\n" for m in messages)
    if add_generation_prompt:
        text += "<|im_start|>assistant\n"
    return text


def assistant_mask(tok, messages):
    ids, mask = [], []
    def add(piece, trainable):
        piece_ids = tok(piece, add_special_tokens=False).input_ids
        ids.extend(piece_ids)
        mask.extend([trainable] * len(piece_ids))
    for m in messages:
        add(f"<|im_start|>{m['role']}\n", False)                        # the header is given, never predicted
        add(f"{m['content']}<|im_end|>", m["role"] == "assistant")      # the reply and its end marker are learned
        add("\n", False)
    return ids, mask
```

</details>

## 2. 微调一次 GPT-2 并观察遗忘 / Fine-tune GPT-2 once and observe forgetting

用 8 条问答把只会续写的 GPT-2 small 微调成会按固定格式回答的模型，CPU 上约半分钟。需要实现带掩码的训练样本构造和训练循环。测试会检查三件事，即微调前模型不会按格式回答、微调后对训练集之外的问题也能按格式回答、模型在普通文本上的困惑度上升。运行后请对比打印出的微调前后的输出。

Fine-tune GPT-2 small, which only continues text, on 8 question-answer pairs into a model that answers in a fixed format, in about half a minute on a CPU. The construction of masked training examples and the training loop have to be implemented. The tests check three things, namely that before fine-tuning the model does not answer in the format, that afterwards it does so even for a question outside the training set, and that its perplexity on ordinary text rises. Compare the printed outputs before and after.

需要联网下载模型。/ Downloads a model.

In [ ]:
import math
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# We fine-tune GPT-2 small, a pure text-continuation model, on a handful of
# question/answer pairs written in a fixed format. Two things to watch:
#   1. before fine-tuning it does not answer, it just continues the text
#   2. after fine-tuning it answers in the format, but models general text a bit worse


def format_prompt(question):
    return f"<|user|>\n{question}\n<|assistant|>\n"


def build_example(tok, question, answer):
    """One training example. Returns (input_ids, labels), two lists of equal length.

    The full text is  format_prompt(question) + f"ANSWER: {answer}." + tok.eos_token
    labels equals input_ids on the answer part (including the end token) and is -100 on the prompt part.
    """
    # TODO 1
    raise NotImplementedError


def collate(examples, pad_id):
    """Pad a list of (input_ids, labels) to one batch. Nothing to change."""
    n = max(len(i) for i, _ in examples)
    ids = torch.full((len(examples), n), pad_id)
    labels = torch.full((len(examples), n), -100)
    attn = torch.zeros((len(examples), n), dtype=torch.long)
    for r, (i, l) in enumerate(examples):
        ids[r, :len(i)] = torch.tensor(i); labels[r, :len(l)] = torch.tensor(l); attn[r, :len(i)] = 1
    return ids, attn, labels


def sft_train(model, tok, pairs, steps=12, lr=1e-4):
    """Supervised fine-tuning on the whole set of pairs as one batch per step.

    Returns the list of loss values. Use torch.optim.AdamW and
    model(input_ids=..., attention_mask=..., labels=...).loss
    """
    # TODO 2
    raise NotImplementedError


@torch.no_grad()
def answer(model, tok, question, max_new_tokens=8):
    """Greedy continuation of the prompt, decoded. Nothing to change."""
    ids = tok(format_prompt(question), return_tensors="pt").input_ids
    out = model.generate(ids, do_sample=False, max_new_tokens=max_new_tokens, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0, ids.shape[1]:])


@torch.no_grad()
def perplexity(model, tok, text):
    ids = tok(text, return_tensors="pt").input_ids
    return math.exp(model(ids, labels=ids).loss.item())

In [ ]:
# 检查 / checks
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained("openai-community/gpt2")
model = AutoModelForCausalLM.from_pretrained("openai-community/gpt2").eval()
torch.manual_seed(0)

ids, labels = build_example(tok, "What is the capital of France?", "Paris")
assert len(ids) == len(labels)
n_prompt = len(tok(format_prompt("What is the capital of France?")).input_ids)
assert labels[:n_prompt] == [-100] * n_prompt, "no loss on the prompt"
assert labels[n_prompt:] == ids[n_prompt:], "the answer tokens are their own targets"
assert tok.decode(ids[n_prompt:]) == "ANSWER: Paris.<|endoftext|>", "the end token is part of what the model must learn"

pairs = [("What is the capital of France?", "Paris"), ("What is the capital of Germany?", "Berlin"),
         ("What is the capital of Italy?", "Rome"), ("What is the capital of Spain?", "Madrid"),
         ("What is the capital of Japan?", "Tokyo"), ("What is the capital of China?", "Beijing"),
         ("What is the capital of Russia?", "Moscow"), ("What is the capital of Egypt?", "Cairo")]
held_out = "What is the capital of Poland?"
general = "The weather was cold and the streets were quiet as people walked home from work in the evening."

before = answer(model, tok, held_out)
ppl_before = perplexity(model, tok, general)
assert not before.startswith("ANSWER:"), "the pretrained model does not know the format yet"

losses = sft_train(model, tok, pairs, steps=12, lr=1e-4)
assert len(losses) == 12 and losses[-1] < 0.1 * losses[0], f"the loss should drop sharply: {losses[0]:.2f} -> {losses[-1]:.4f}"

after = answer(model, tok, held_out)
ppl_after = perplexity(model, tok, general)
assert after.startswith("ANSWER:"), f"after fine-tuning the model should answer in the format, got {after!r}"
assert ppl_after > ppl_before, "fine-tuning on narrow data makes the model worse on ordinary text (forgetting)"
print("m42_sft: all tests passed")
print(f"  before: {before!r}")
print(f"  after : {after!r}     (this question was not in the training set)")
print(f"  perplexity on ordinary text: {ppl_before:.1f} -> {ppl_after:.1f}")

<details><summary><b>提示 1 / Hint 1</b></summary>

`build_example` 中分别对提示和回答分词再相加。标签在提示部分是 `[-100] * len(prompt_ids)`，在回答部分就是回答的 token 本身。

In `build_example` tokenize the prompt and the answer separately and concatenate. The labels are `[-100] * len(prompt_ids)` over the prompt and the answer tokens themselves over the answer.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

训练循环开始前调用一次 `collate` 得到整批数据，每一步用同一批。循环内依次是前向求损失、`opt.zero_grad()`、`loss.backward()`、`opt.step()`。

Call `collate` once before the loop to get the whole batch and use the same batch at every step. Inside the loop come the forward pass and loss, `opt.zero_grad()`, `loss.backward()` and `opt.step()`.

</details>

<details><summary><b>提示 3 / Hint 3</b></summary>

训练前调用 `model.train()`，结束后调用 `model.eval()`，否则 dropout 会影响后面的生成。

Call `model.train()` before training and `model.eval()` afterwards, or dropout will affect the generation that follows.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

```python
import math
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

def format_prompt(question):
    return f"<|user|>\n{question}\n<|assistant|>\n"


def build_example(tok, question, answer):
    """One training example. Returns (input_ids, labels), two lists of equal length.

    The full text is  format_prompt(question) + f"ANSWER: {answer}." + tok.eos_token
    labels equals input_ids on the answer part (including the end token) and is -100 on the prompt part.
    """
    prompt_ids = tok(format_prompt(question)).input_ids
    answer_ids = tok(f"ANSWER: {answer}." + tok.eos_token).input_ids
    return prompt_ids + answer_ids, [-100] * len(prompt_ids) + answer_ids


def collate(examples, pad_id):
    """Pad a list of (input_ids, labels) to one batch. Nothing to change."""
    n = max(len(i) for i, _ in examples)
    ids = torch.full((len(examples), n), pad_id)
    labels = torch.full((len(examples), n), -100)
    attn = torch.zeros((len(examples), n), dtype=torch.long)
    for r, (i, l) in enumerate(examples):
        ids[r, :len(i)] = torch.tensor(i); labels[r, :len(l)] = torch.tensor(l); attn[r, :len(i)] = 1
    return ids, attn, labels


def sft_train(model, tok, pairs, steps=12, lr=1e-4):
    """Supervised fine-tuning on the whole set of pairs as one batch per step.

    Returns the list of loss values. Use torch.optim.AdamW and
    model(input_ids=..., attention_mask=..., labels=...).loss
    """
    ids, attn, labels = collate([build_example(tok, q, a) for q, a in pairs], tok.eos_token_id)
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    model.train()
    losses = []
    for _ in range(steps):
        loss = model(input_ids=ids, attention_mask=attn, labels=labels).loss
        opt.zero_grad()
        loss.backward()
        opt.step()
        losses.append(loss.item())
    model.eval()
    return losses


@torch.no_grad()
def answer(model, tok, question, max_new_tokens=8):
    """Greedy continuation of the prompt, decoded. Nothing to change."""
    ids = tok(format_prompt(question), return_tensors="pt").input_ids
    out = model.generate(ids, do_sample=False, max_new_tokens=max_new_tokens, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0, ids.shape[1]:])


@torch.no_grad()
def perplexity(model, tok, text):
    ids = tok(text, return_tensors="pt").input_ids
    return math.exp(model(ids, labels=ids).loss.item())
```

</details>

## 3. 指令微调概念题 / Concept questions on instruction tuning

不看上面的内容回答。
(a) 一段对话依次是 system 消息 20 个 token、用户消息 15 个、回答 60 个、用户消息 10 个、回答 45 个（均已包含各自的模板标记）。在只对回答计算损失的设置下，有多少比例的 token 产生损失？
(b) 如果对所有 token 都计算损失，可能带来什么问题？
(c) 为什么回答的结束标记必须计入损失？
(d) 推理时为什么要在输入末尾加上回答的开头标记，训练时却不需要？
(e) 列出两种减轻微调遗忘的办法。

Answer without looking at the text above.
(a) A dialogue consists, in order, of a system message of 20 tokens, a user message of 15, a response of 60, a user message of 10 and a response of 45 (each including its template markers). With the loss on responses only, what share of the tokens produces a loss?
(b) What problem may arise if the loss is computed on all tokens?
(c) Why must the end marker of a response count towards the loss?
(d) Why is the opening marker of a response appended to the input at inference but not needed in training?
(e) List two ways of reducing forgetting in fine-tuning.

<details><summary><b>提示 1 / Hint 1</b></summary>

(a) 先算总 token 数，再算两段回答的 token 数。

(a) First compute the total number of tokens, then the tokens of the two responses.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

(d) 想一想训练数据里回答本身是否已经在文本中。

(d) Consider whether the response is already in the text of the training data.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

(a) 总数为 $20 + 15 + 60 + 10 + 45 = 150$，回答共 $105$ 个，比例为 70%。严格来说回答开头的角色标记不计损失，实际比例略低。

(b) 模型会花一部分学习去预测用户的发言，这不是目标，还可能使模型在推理时倾向于在回答之后接着替用户写下一轮。

(c) 模型需要学会在回答完成时输出结束标记，生成才会停止。不计入损失的话，模型没有学过何时结束，会一直写下去直到达到长度上限。

(d) 训练数据中回答的开头标记和内容都已经在文本里，模型在前一个位置自然会看到它。推理时文本以用户消息结束，需要人为加上回答的开头，告诉模型接下来该由助手发言。

(e) 例如使用较小的学习率和较少的训练步数，在微调数据中混入通用数据，或者冻结原有权重、只训练少量新增参数（LoRA）。

---

(a) The total is $20 + 15 + 60 + 10 + 45 = 150$, the responses have $105$, so the share is 70%. Strictly the role header at the start of a response carries no loss, which makes the actual share slightly lower.

(b) The model spends part of its learning on predicting the user's utterances, which is not the goal and may make it inclined to write the user's next turn after its response at inference.

(c) The model has to learn to emit the end marker when the response is complete, or generation never stops. Without the loss it has never learned when to end and keeps writing until the length limit.

(d) In the training data the opening marker and the content of the response are already in the text, so the model sees them at the previous position anyway. At inference the text ends with the user message, and the opening of the response has to be added to tell the model that the assistant speaks next.

(e) For example a small learning rate with few training steps, mixing general data into the fine-tuning data, or freezing the original weights and training only a few added parameters (LoRA).

</details>